# Scopus Data Processing and Deduplication Pipeline

This notebook processes multiple CSV files exported from Scopus, corresponding to different search strategies.

The workflow follows this sequence:

1. Load and merge all CSV files into a single dataset
2. Standardize key bibliographic fields
3. Identify and remove duplicate records
4. Apply document-level exclusions
5. Save two CSV files:
   - A cleaned dataset for review
   - A formatted dataset for Biblioshiny

Record counts are reported at each stage to document the number of records loaded, removed, and retained.


In [ ]:
from pathlib import Path
import html
import pandas as pd


## Paths and Output Files

Set the project folder and the names of the input and output folders. The input folder should contain the Scopus CSV files exported from the search strategies.


In [ ]:
project_folder = Path(r"x:/path/to/project")
input_folder = project_folder / "input_csv"
output_folder = project_folder / "output_csv"

file_pattern = "*.csv"
cleaned_file = "scopus_cleaned.csv"
biblioshiny_file = "scopus_biblioshiny.csv"

required_fields = ["Title", "DOI", "Author Keywords", "Index Keywords"]
remove_article_in_press = True


## Load and Merge CSV Files

All CSV files in the input folder are loaded and merged into a single dataset before cleaning, deduplication, or exclusion.


In [ ]:
files = sorted(input_folder.glob(file_pattern))

if not files:
    raise FileNotFoundError("No CSV files were found in the input folder.")

tables = []

print("Input files:")
for file in files:
    table = pd.read_csv(file, dtype=str, encoding="utf-8-sig", keep_default_na=False)
    table["source_file"] = file.name
    tables.append(table)
    print("-", file.name, "| Records:", len(table))

data = pd.concat(tables, ignore_index=True)

records_loaded = sum(len(table) for table in tables)
records_merged = len(data)

print("")
print("Merge summary")
print("Files merged:", len(files))
print("Records loaded:", records_loaded)
print("Records after merging:", records_merged)


## Standardize Text Fields

Text values are standardized before filtering and deduplication. This step removes HTML marks, extra spaces, and repeated empty-value labels.


In [ ]:
empty_values = {"", " ", "na", "n/a", "nan", "none", "null"}

records_before_text_cleaning = len(data)

for column in data.columns:
    data[column] = data[column].apply(lambda value: html.unescape(str(value)) if pd.notna(value) else "")
    data[column] = data[column].str.replace(r"<[^>]+>", " ", regex=True)
    data[column] = data[column].str.replace(r"\s+", " ", regex=True).str.strip()
    data.loc[data[column].str.lower().isin(empty_values), column] = pd.NA

records_after_text_cleaning = len(data)
records_removed_text_cleaning = records_before_text_cleaning - records_after_text_cleaning

print("Text standardization summary")
print("Records before text standardization:", records_before_text_cleaning)
print("Records removed during text standardization:", records_removed_text_cleaning)
print("Records after text standardization:", records_after_text_cleaning)


## Remove Duplicate Records

Duplicate records are checked in three passes: EID, DOI, and title with year. Records without the corresponding identifier are kept for the following checks.


In [ ]:
clean = data.copy()

before_eid = len(clean)
removed_by_eid = 0

if "EID" in clean.columns:
    clean["clean_eid"] = clean["EID"].str.lower().str.strip()
    clean.loc[clean["clean_eid"].str.lower().isin(empty_values), "clean_eid"] = pd.NA

    has_eid = clean["clean_eid"].notna()
    clean = pd.concat([
        clean.loc[has_eid].drop_duplicates(subset=["clean_eid"], keep="first"),
        clean.loc[~has_eid]
    ], ignore_index=True)

removed_by_eid = before_eid - len(clean)

before_doi = len(clean)
removed_by_doi = 0

if "DOI" in clean.columns:
    clean["clean_doi"] = clean["DOI"].str.lower().str.strip()
    clean["clean_doi"] = clean["clean_doi"].str.replace(r"^https?://(dx\.)?doi\.org/", "", regex=True)
    clean["clean_doi"] = clean["clean_doi"].str.replace(r"^doi:\s*", "", regex=True)
    clean["clean_doi"] = clean["clean_doi"].str.rstrip(".")
    clean.loc[clean["clean_doi"].str.lower().isin(empty_values), "clean_doi"] = pd.NA

    has_doi = clean["clean_doi"].notna()
    clean = pd.concat([
        clean.loc[has_doi].drop_duplicates(subset=["clean_doi"], keep="first"),
        clean.loc[~has_doi]
    ], ignore_index=True)

removed_by_doi = before_doi - len(clean)

before_title_year = len(clean)
removed_by_title_year = 0

if "Title" in clean.columns and "Year" in clean.columns:
    clean["clean_title"] = clean["Title"].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
    clean["clean_year"] = clean["Year"].astype(str).str.extract(r"(\d{4})", expand=False)

    has_title_year = clean["clean_title"].notna() & clean["clean_year"].notna()
    clean = pd.concat([
        clean.loc[has_title_year].drop_duplicates(subset=["clean_title", "clean_year"], keep="first"),
        clean.loc[~has_title_year]
    ], ignore_index=True)

removed_by_title_year = before_title_year - len(clean)
records_removed_duplicates = removed_by_eid + removed_by_doi + removed_by_title_year

print("Deduplication summary")
print("Records before deduplication:", records_after_text_cleaning)
print("Records removed by EID:", removed_by_eid)
print("Records removed by DOI:", removed_by_doi)
print("Records removed by title and year:", removed_by_title_year)
print("Total records removed as duplicates:", records_removed_duplicates)
print("Records after deduplication:", len(clean))


## Apply Document-Level Exclusions

Records without the required bibliographic fields are excluded. Article in press records are excluded when the column is available.


In [ ]:
missing_fields = [field for field in required_fields if field not in clean.columns]

if missing_fields:
    raise KeyError("Missing required fields: " + ", ".join(missing_fields))

before_required_fields = len(clean)
clean = clean.dropna(subset=required_fields).copy()
removed_by_required_fields = before_required_fields - len(clean)

before_publication_stage = len(clean)
removed_by_publication_stage = 0

if remove_article_in_press and "Publication Stage" in clean.columns:
    publication_stage = clean["Publication Stage"].str.lower().str.strip()
    clean = clean.loc[publication_stage.ne("article in press")].copy()
    removed_by_publication_stage = before_publication_stage - len(clean)

records_removed_exclusions = removed_by_required_fields + removed_by_publication_stage

print("Exclusion summary")
print("Records before exclusions:", before_required_fields)
print("Records removed due to missing required fields:", removed_by_required_fields)
print("Records removed as article in press:", removed_by_publication_stage)
print("Total records removed during exclusions:", records_removed_exclusions)
print("Records after exclusions:", len(clean))


## Save CSV Files

Two CSV files are saved: one cleaned dataset and one dataset formatted for Biblioshiny.


In [ ]:
output_folder.mkdir(parents=True, exist_ok=True)

helper_columns = ["clean_eid", "clean_doi", "clean_title", "clean_year"]

cleaned_data = clean.drop(columns=helper_columns, errors="ignore").copy()
biblioshiny_data = cleaned_data.drop(columns=["source_file"], errors="ignore").copy()

cleaned_path = output_folder / cleaned_file
biblioshiny_path = output_folder / biblioshiny_file

cleaned_data.to_csv(cleaned_path, index=False, encoding="utf-8-sig")
biblioshiny_data.to_csv(biblioshiny_path, index=False, encoding="utf-8-sig")

print("Output summary")
print("Cleaned CSV:", cleaned_path)
print("Biblioshiny CSV:", biblioshiny_path)
print("Records saved in cleaned CSV:", len(cleaned_data))
print("Records saved in Biblioshiny CSV:", len(biblioshiny_data))


## Processing Summary

The following counts summarize the complete workflow from the merged Scopus exports to the final retained records.


In [ ]:
print("Processing summary")
print("Files merged:", len(files))
print("Records loaded:", records_loaded)
print("Records after merging:", records_merged)
print("Records removed during text standardization:", records_removed_text_cleaning)
print("Records removed as duplicates:", records_removed_duplicates)
print("Records removed during exclusions:", records_removed_exclusions)
print("Final records retained:", len(cleaned_data))
